# S1 — Decoder Oracle: is the JEPA latent invertible to RGB?**Definition of done for this notebook:** a number, with a null, for how much of aframe each candidate latent retains — and a watchable reconstructed video of theheld-out block if the answer is yes.This is the **oracle** condition: decode the *real encoded* latent of a real frame,never a predicted one. If the real latent cannot be inverted, no prediction qualitycan rescue the pipeline; if it can, then and only then is the predicted latent worthdecoding. Keeping these two questions apart is the whole point — the previousdiffusion-decoder arc conflated them and landed at 1/300 on the decisive check, ator below the ~4/300 chance rate.---## The correction that made this notebook necessaryNotes filed on 2026-08-27 claimed:> 196 tokens × 768 dims = 150,528 floats = exactly a 224² image's 150,528 values.> Ratio exactly 1.0. No compression bottleneck.**That is not the pipeline.** `FarlFrozenFaceEncoder.forward()` ends with```pythonpatch_mean = x[:, 1:].mean(dim=1).float()   # (N, 768)out.last_hidden_state = patch_mean.unsqueeze(1)```and `JEPA_Latent_Probing.ipynb` cells 65/73/89 do the same(`hidden[:, 1:].mean(dim=1)`, printed as "patch-mean"). The real latent is **768floats per frame**, i.e. **196× compression**, and the JEPA state that the predictoractually operates on is the 256-d `projector` output — **588× compression**.That puts the decoder in the pooled-vector regime (RCDM, unCLIP) rather than thespatial-token regime (RAE, VideoRAE). RAE's rFID numbers — DINOv2-B 0.49, SigLIP2-B0.53, MAE-B 0.16, against SD-VAE's 0.62 — are for *spatial tokens*, and do nottransfer to a pooled vector. So this notebook measures all three, because extractingdense tokens is a one-line change and guessing is what got us here:| representation | dims/frame | compression vs 224²×3 | who uses it ||---|---|---|---|| `pooled_state` | 256 | 588× | the JEPA predictor (Run D/E/F) || `farl_patch_mean` | 768 | 196× | `FarlFrozenFaceEncoder` output || `farl_dense` | 196 × 768 = 150,528 | 1.0× | nothing yet — the assumed one |If `farl_dense` inverts well and `farl_patch_mean` does not, the finding is concreteand actionable: **the mean-pool is the bottleneck**, and the fix is one line in theencoder plus a wider projector. That is a better outcome than either extreme.## Order of operations1. **Path 0** — closed-form ridge, no training, minutes. An *upper bound on linear   invertibility* per representation. Cheap enough to run first and it kills bad   options before any GPU-hours are spent.2. **Path 1** — small trained decoders, one per representation.3. **The decisive check** — N-way mouth-region matching against a binomial null.4. **Video** — reconstruct the held-out block and write an mp4.Run `S0_Instrument_And_Audio_Diagnostics.ipynb` first; this notebook reuses itsFaRL cache, its split, and its Run F checkpoint.

## 0. Setup — shared with S0

In [ ]:
from google.colab import drivedrive.mount('/content/drive')

In [ ]:
import os, sys, glob, json, time, gc, math, tarfile, shutilimport numpy as npimport torchimport torch.nn as nnimport torch.nn.functional as Fimport cv2import matplotlib.pyplot as pltfrom einops import rearrangefrom tqdm.auto import tqdmCODE_BASE = '/content/drive/MyDrive/Data/le-wm'assert os.path.isdir(CODE_BASE), f'{CODE_BASE} not found'os.chdir(CODE_BASE); sys.path.insert(0, CODE_BASE)from module import MLP, ARPredictordevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print('device:', device)IDENTITY   = 'May'DATA_ROOT  = os.path.join(CODE_BASE, 'data')LOCAL_ROOT = '/content/local_data'CACHE_DIR  = os.path.join(CODE_BASE, 'study_cache');   os.makedirs(CACHE_DIR, exist_ok=True)PREREG_DIR = os.path.join(CODE_BASE, 'prereg');        os.makedirs(PREREG_DIR, exist_ok=True)RESULTS_DIR= os.path.join(CODE_BASE, 'study_results'); os.makedirs(RESULTS_DIR, exist_ok=True)CKPT_DIR   = os.path.join(CODE_BASE, 'checkpoints')LOCAL_CACHE= '/content/dense_cache'; os.makedirs(LOCAL_CACHE, exist_ok=True)STUDY_ID = 'S1_decoder_oracle'RUN_TAG  = 'runF_trainblock_only'      # the S0 checkpoint whose projector defines pooled_stateEMBED_DIM, HIDDEN_DIM, HISTORY_SIZE, AUDIO_DIM = 256, 256, 4, 512IMG = 224; PATCH = 16; GRID = IMG // PATCH      # 14 x 14 = 196 patchesSEED = 0torch.manual_seed(SEED); np.random.seed(SEED)

## 1. Pre-registrationSame write-once discipline as S0. The primary endpoint is the **decisive check**, nota pixel metric — L1 and PSNR reward a blurry average face, which is exactly thefailure mode that must not be allowed to read as success.

In [ ]:
PREREG_PATH = os.path.join(PREREG_DIR, f'{STUDY_ID}.json')PREREG = {    'study_id': STUDY_ID,    'question': 'Which latent, if any, can be inverted to a frame that carries the '                'per-frame mouth configuration?',    'condition': 'ORACLE - decode the real encoded latent of a real held-out frame. '                 'Predicted latents are out of scope for this notebook.',    'representations': {        'pooled_state':    {'dims': 256,    'compression': 588.0},        'farl_patch_mean': {'dims': 768,    'compression': 196.0},        'farl_dense':      {'dims': 150528, 'compression': 1.0},    },    'split': {'kind': 'contiguous official SyncTalk split', 'train': '0..5519', 'test': '5520..6072'},    'primary_endpoint': {        'name': 'mouth_region_nway_matching_heldout',        'definition': ('For each of N_QUERIES=300 held-out frames, crop the mouth region of the '                       'reconstruction and match it (by L2 on the mouth crop) against N_WAY=75 '                       'candidate REAL mouth crops: the true frame plus 74 distractors drawn from '                       'the held-out block. Count top-1 hits.'),        'null': 'Binomial(n=300, p=1/75): mean 4.000, SD 1.987',        'harness': ('The true candidate is placed at a RANDOM position among the 75, not first: '                    'with it always first, argmin first-wins tie-breaking hands ties to the '                    'correct answer and a mean-image control reaches 3x chance. Primary uses '                    'exclude_radius=0 (adjacent frames stay in the distractor pool); an '                    'exclude_radius=5 variant is reported as a secondary.'),        'critical_values': {'p<0.05': '>= 8', 'p<0.01': '>= 10', 'p<0.001': '>= 12'},        'pass': '>= 12 hits (p < 0.001)',        'weak': '8..11 hits (p < 0.05)',        'fail': '<= 7 hits',        'power_note': ('MDE at 80% power is 2.54x chance = ~10 hits. Power is only 25% at 1.5x '                       'chance and 55% at 2x. A null result at this N does not distinguish '                       '"no effect" from "small effect" - report the CI, not just the verdict. '                       'P(1 <= X <= 8) = 0.9616 under the null, so a single-digit count is '                       'entirely ordinary noise.'),        'do_not': ('Do not convert the count to a z-score as (x-4)/1.987 and read a normal tail. '                   'That was done before and is anti-conservative by up to 30x at these counts. '                   'Use the exact binomial survival function, which this notebook computes.'),    },    'secondary_endpoints': [        'Path-0 closed-form ridge R^2 to downsampled pixels, per representation',        'Path-1 trained decoder L1 and PSNR on held-out frames, per representation',        'ratio to the mean-image baseline (predict the train-block average frame)',        'latent-swap control: decode frame j latent, score against frame i',    ],    'controls': {        'mean_image': 'predict the training-block mean frame regardless of input - the number '                      'every pixel metric must beat by a wide margin to mean anything',        'random_feature': 'same-dimensional Gaussian noise in place of the latent, same fit',        'latent_swap': 'decode a DIFFERENT frame latent - if the decisive check score barely '                       'drops, the decoder is not input-dependent',        'shuffled_target': 'fit the decoder against permuted targets - must collapse to chance',    },    'stop_rule': ('If ALL THREE representations fail the decisive check, the finding is that '                  'this encoder/latent family is not invertible at this budget - report it and '                  'stop, do not escalate to a diffusion decoder. The previous diffusion arc '                  'already produced 1/300 and cost weeks.'),    'registered_at_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),}if os.path.isfile(PREREG_PATH):    with open(PREREG_PATH) as f: PREREG = json.load(f)    print(f'Loaded EXISTING pre-registration ({PREREG["registered_at_utc"]}) - not overwritten.')else:    with open(PREREG_PATH, 'w') as f: json.dump(PREREG, f, indent=2)    print(f'Pre-registration WRITTEN to {PREREG_PATH}')N_QUERIES, N_WAY = 300, 75from scipy.stats import binomprint(f'null: Binomial({N_QUERIES}, 1/{N_WAY})  mean {N_QUERIES/N_WAY:.3f}  '      f'SD {math.sqrt(N_QUERIES*(1/N_WAY)*(1-1/N_WAY)):.3f}')for a in (0.05, 0.01, 0.001):    k = int(binom.isf(a, N_QUERIES, 1/N_WAY)) + 1    print(f'  critical value for p<{a}: >= {k} hits')

## 2. Data, split, and the frames themselves

In [ ]:
os.makedirs(LOCAL_ROOT, exist_ok=True)def ensure_local_identity(name):    local = os.path.join(LOCAL_ROOT, name)    if os.path.isdir(os.path.join(local, 'gt_imgs')):        return local    tar = os.path.join(DATA_ROOT, f'{name}_train.tar')    if os.path.isfile(tar):        with tarfile.open(tar) as t: t.extractall(LOCAL_ROOT)    else:        shutil.copytree(os.path.join(DATA_ROOT, name), local)    return localdef sorted_numeric_jpgs(dir_path):    kept = []    for p in glob.glob(os.path.join(dir_path, '*.jpg')):        stem = os.path.splitext(os.path.basename(p))[0]        try: int(stem); kept.append(p)        except ValueError: pass    kept.sort(key=lambda p: int(os.path.splitext(os.path.basename(p))[0]))    return keptLOCAL_PATH  = ensure_local_identity(IDENTITY)FRAME_PATHS = sorted_numeric_jpgs(os.path.join(LOCAL_PATH, 'gt_imgs'))N_FRAMES    = len(FRAME_PATHS)def official_split(drive_identity_dir, n_frames):    tr = os.path.join(drive_identity_dir, 'transforms_train.json')    va = os.path.join(drive_identity_dir, 'transforms_val.json')    if os.path.isfile(tr) and os.path.isfile(va):        with open(tr) as f: t = json.load(f)['frames']        with open(va) as f: v = json.load(f)['frames']        return (np.array(sorted(int(x['img_id']) for x in t if int(x['img_id']) < n_frames)),                np.array(sorted(int(x['img_id']) for x in v if int(x['img_id']) < n_frames)))    cut = int(n_frames * 10 / 11)    return np.arange(cut), np.arange(cut, n_frames)TRAIN_IDX, TEST_IDX = official_split(os.path.join(DATA_ROOT, IDENTITY), N_FRAMES)assert TRAIN_IDX.max() < TEST_IDX.min() and np.all(np.diff(TEST_IDX) == 1)print(f'{N_FRAMES} frames | train {TRAIN_IDX.min()}..{TRAIN_IDX.max()} ({len(TRAIN_IDX)}) | '      f'test {TEST_IDX.min()}..{TEST_IDX.max()} ({len(TEST_IDX)})')# Subsample the train block for the dense-token cache (196x768 fp16 = 301 KB/frame).# Every 2nd train frame + ALL test frames.DENSE_TRAIN = TRAIN_IDX[::2]DENSE_IDX   = np.concatenate([DENSE_TRAIN, TEST_IDX])print(f'dense cache: {len(DENSE_IDX)} frames '      f'({len(DENSE_IDX)*196*768*2/1e9:.2f} GB fp16 on local disk)')

In [ ]:
def load_pixels(idxs, size=IMG):    '''Project-canonical preprocessing: BGR->RGB, squash-resize 224, /255. No crop.'''    imgs = []    for i in idxs:        img = cv2.imread(FRAME_PATHS[int(i)])        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)        img = cv2.resize(img, (size, size))        imgs.append(img)    arr = np.stack(imgs).astype(np.uint8)    return torch.from_numpy(arr.astype(np.float32) / 255.0).permute(0, 3, 1, 2), arr# uint8 pixel cache for every frame we will score (test block + the dense train subset)PIX_CACHE_PATH = os.path.join(LOCAL_CACHE, 'pixels_u8.npy')if os.path.isfile(PIX_CACHE_PATH):    PIX = np.load(PIX_CACHE_PATH, mmap_mode='r')else:    PIX = np.empty((len(DENSE_IDX), IMG, IMG, 3), dtype=np.uint8)    for b0 in tqdm(range(0, len(DENSE_IDX), 64), desc='pixel cache'):        sl = slice(b0, min(b0 + 64, len(DENSE_IDX)))        _, arr = load_pixels(DENSE_IDX[sl])        PIX[sl] = arr    np.save(PIX_CACHE_PATH, PIX)    PIX = np.load(PIX_CACHE_PATH, mmap_mode='r')POS = {int(f): i for i, f in enumerate(DENSE_IDX)}      # frame id -> row in PIX / DENSEprint('pixel cache:', PIX.shape, PIX.dtype)

## 3. The three representations`farl_patch_mean` comes from S0's cache. `farl_dense` and `pooled_state` are builthere. The dense pass is the expensive one (~3300 frames through ViT-B/16, a fewminutes on a T4) and is written to **local** disk, not Drive — 1 GB through the FUSEmount is not worth it and the cache is cheap to rebuild.

In [ ]:
!pip install -q ftfy regex!pip install -q git+https://github.com/openai/CLIP.git

In [ ]:
import urllib.requestclass FarlEncoder(nn.Module):    '''Same weights and same preprocessing as FarlFrozenFaceEncoder, but exposes the    FULL token sequence so both the patch-mean and the dense tokens come from one pass.'''    URL = ('https://github.com/FacePerceiver/FaRL/releases/download/pretrained_weights/'           'FaRL-Base-Patch16-LAIONFace20M-ep16.pth')    def __init__(self, checkpoint_dir=None):        super().__init__()        import clip        d = checkpoint_dir or os.path.join(CODE_BASE, 'probing_cache', 'farl')        os.makedirs(d, exist_ok=True)        p = os.path.join(d, 'FaRL-Base-Patch16-LAIONFace20M-ep16.pth')        if not os.path.isfile(p):            urllib.request.urlretrieve(self.URL, p)        m, _ = clip.load('ViT-B/16', device='cpu')        m.load_state_dict(torch.load(p, map_location='cpu')['state_dict'], strict=False)        self.visual = m.visual.float()        for q in self.visual.parameters(): q.requires_grad_(False)        self.visual.eval()    def train(self, mode=True): return super().train(False)    @torch.no_grad()    def forward(self, pixels):        v = self.visual        x = pixels.type(v.conv1.weight.dtype)        x = v.conv1(x).reshape(pixels.shape[0], -1, GRID * GRID).permute(0, 2, 1)        cls = v.class_embedding.to(x.dtype) + torch.zeros(x.shape[0], 1, x.shape[-1],                                                          dtype=x.dtype, device=x.device)        x = torch.cat([cls, x], 1) + v.positional_embedding.to(x.dtype)        x = v.ln_pre(x).permute(1, 0, 2)        x = v.transformer(x).permute(1, 0, 2)        x = v.ln_post(x).float()                 # (N, 197, 768), ln_post on ALL tokens        return x[:, 1:], x[:, 1:].mean(1)        # dense (N,196,768), patch_mean (N,768)

In [ ]:
DENSE_PATH = os.path.join(LOCAL_CACHE, 'farl_dense_f16.npy')MEAN_PATH  = os.path.join(LOCAL_CACHE, 'farl_patchmean.npy')if os.path.isfile(DENSE_PATH) and os.path.isfile(MEAN_PATH):    DENSE = np.load(DENSE_PATH, mmap_mode='r')    PMEAN = np.load(MEAN_PATH)    print('loaded dense cache', DENSE.shape)else:    enc = FarlEncoder().to(device).eval()    DENSE = np.lib.format.open_memmap(DENSE_PATH, mode='w+', dtype=np.float16,                                      shape=(len(DENSE_IDX), GRID * GRID, 768))    PMEAN = np.empty((len(DENSE_IDX), 768), dtype=np.float32)    B = 16    for b0 in tqdm(range(0, len(DENSE_IDX), B), desc='FaRL dense'):        sl = slice(b0, min(b0 + B, len(DENSE_IDX)))        px, _ = load_pixels(DENSE_IDX[sl])        d, m = enc(px.to(device))        DENSE[sl] = d.cpu().numpy().astype(np.float16)        PMEAN[sl] = m.cpu().numpy()    DENSE.flush(); np.save(MEAN_PATH, PMEAN)    DENSE = np.load(DENSE_PATH, mmap_mode='r')    del enc; gc.collect()    if torch.cuda.is_available(): torch.cuda.empty_cache()print('DENSE', DENSE.shape, DENSE.dtype, '| PMEAN', PMEAN.shape)# consistency: the cached mean must equal the mean of the cached dense tokenschk = np.abs(np.asarray(DENSE[:8], dtype=np.float32).mean(1) - PMEAN[:8]).max()print(f'max |mean(dense) - patch_mean| = {chk:.2e}  (fp16 rounding only)')assert chk < 1e-2, 'dense and patch-mean caches disagree'

In [ ]:
# --- pooled_state: the 256-d JEPA state, via Run F's trained projector ---------class CachedProjector(nn.Module):    def __init__(self, feat_dim=768, embed_dim=EMBED_DIM):        super().__init__()        self.projector = MLP(input_dim=feat_dim, output_dim=embed_dim,                             hidden_dim=1024, norm_fn=nn.BatchNorm1d)CKPT_PATH = os.path.join(CKPT_DIR, f'{RUN_TAG}_best.pth')if os.path.isfile(CKPT_PATH):    ck = torch.load(CKPT_PATH, map_location='cpu')    proj = CachedProjector().to(device)    sd = {k: v for k, v in ck['model_state_dict'].items() if k.startswith('projector.')}    missing, unexpected = proj.load_state_dict(sd, strict=False)    assert not [m for m in missing if m.startswith('projector.')], missing    proj.eval()    with torch.no_grad():        POOLED = np.concatenate([            proj.projector(torch.from_numpy(PMEAN[b0:b0+512]).to(device)).cpu().numpy()            for b0 in range(0, len(PMEAN), 512)])    print(f'pooled_state from {RUN_TAG} epoch {ck["epoch"]}: {POOLED.shape}')else:    POOLED = None    print(f'WARNING: {CKPT_PATH} not found - run S0 first. Skipping the pooled_state row; '          f'the two FaRL representations still run.')

## 4. The mouth region, derived from the dataThe decisive check needs a mouth box. Rather than adding a landmark dependency, takethe highest-temporal-variance region of the frame: with a fixed camera and a talkinghead, that *is* the mouth. The box is computed on the **train block only** and printedso it can be sanity-checked by eye, with a manual override right below it.

In [ ]:
n_probe = min(600, len(DENSE_TRAIN))probe_rows = [POS[int(f)] for f in DENSE_TRAIN[np.linspace(0, len(DENSE_TRAIN)-1, n_probe).astype(int)]]stack = np.asarray(PIX[sorted(probe_rows)], dtype=np.float32) / 255.0std_map = stack.std(0).mean(-1)                      # (224, 224)sm = cv2.GaussianBlur(std_map, (0, 0), 6)thr = np.percentile(sm, 99.0)ys, xs = np.where(sm >= thr)cy, cx = int(np.median(ys)), int(np.median(xs))H = W = 64MOUTH = (max(0, cy - H // 2), max(0, cx - W // 2))MOUTH = (min(MOUTH[0], IMG - H), min(MOUTH[1], IMG - W))# MOUTH = (140, 84)   # <-- manual override if the automatic box is wrongprint(f'mouth box (top-left): {MOUTH}, size {H}x{W}')fig, ax = plt.subplots(1, 3, figsize=(11, 3.6))ax[0].imshow(stack[len(stack)//2]); ax[0].set_title('a train frame')ax[1].imshow(sm, cmap='magma');     ax[1].set_title('temporal std (blurred)')ov = stack[len(stack)//2].copy()ov[MOUTH[0]:MOUTH[0]+H, MOUTH[1]:MOUTH[1]+W] *= 0.45ax[2].imshow(ov); ax[2].set_title('mouth box')for a in ax: a.axis('off')plt.tight_layout(); plt.show()print('If the darkened box is not on the mouth, set MOUTH by hand above and re-run this cell.')def mouth_crop(imgs):    '''imgs: (N, H, W, 3) float in [0,1] -> (N, 64*64*3)'''    r, c = MOUTH    return imgs[:, r:r+H, c:c+W, :].reshape(len(imgs), -1)

## 5. Path 0 — closed-form linear invertibility (no training)An upper bound on what a *linear* map can recover, per representation. Cheap, exact,and it has no training tricks to argue about. Two probes:- **global** — latent → 32×32×3 downsampled frame (ridge, L2 chosen by 5-fold CV  inside the train block). Applies to all three representations.- **per-patch** — for `farl_dense` only: each of the 196 tokens → *its own* 16×16×3  pixel block. 196 independent 768→768 ridges. This is the spatial-token regime the  RAE literature is actually about, and it is where dense tokens should win if they  win anywhere.Both are reported as R² against the **mean-image baseline**, so "predicts the averageface" scores 0 by construction.

In [ ]:
def ridge_fit(X, Y, l2=1.0):    X = np.asarray(X, np.float64); Y = np.asarray(Y, np.float64)    xm, ym = X.mean(0, keepdims=True), Y.mean(0, keepdims=True)    Xc, Yc = X - xm, Y - ym    W = np.linalg.solve(Xc.T @ Xc + l2 * np.eye(Xc.shape[1]), Xc.T @ Yc)    return W, ym - xm @ Wdef ridge_predict(X, W, b):    return np.asarray(X, np.float64) @ W + bdef r2_vs_mean(pred, Y, Y_train_mean):    '''1 - SSE(pred) / SSE(mean-image). 0 = no better than the average frame.'''    sse = ((pred - Y) ** 2).sum()    sst = ((Y_train_mean[None] - Y) ** 2).sum()    return float(1 - sse / sst)def cv_l2(X, Y, grid, k=5, seed=SEED):    n = len(X); idx = np.random.default_rng(seed).permutation(n)    folds = np.array_split(idx, k)    best, best_s = None, np.inf    for l2 in grid:        s = 0.0        for i in range(k):            va = folds[i]; tr = np.concatenate([folds[j] for j in range(k) if j != i])            mu, sd = X[tr].mean(0, keepdims=True), X[tr].std(0, keepdims=True) + 1e-6            W, b = ridge_fit((X[tr]-mu)/sd, Y[tr], l2)            s += ((ridge_predict((X[va]-mu)/sd, W, b) - Y[va]) ** 2).mean()        if s < best_s: best_s, best = s, l2    return bestL2_GRID = list(np.logspace(0, 8, 9))TR_ROWS = np.array([POS[int(f)] for f in DENSE_TRAIN])TE_ROWS = np.array([POS[int(f)] for f in TEST_IDX])print(f'path-0: {len(TR_ROWS)} train rows, {len(TE_ROWS)} test rows')

In [ ]:
# --- global probe: latent -> 32x32x3 -----------------------------------------SMALL = 32def small_targets(rows):    out = np.empty((len(rows), SMALL * SMALL * 3), np.float32)    for i, r in enumerate(rows):        out[i] = cv2.resize(np.asarray(PIX[r]), (SMALL, SMALL)).astype(np.float32).ravel() / 255.0    return outY_tr, Y_te = small_targets(TR_ROWS), small_targets(TE_ROWS)Y_tr_mean = Y_tr.mean(0)# farl_dense is deliberately NOT in the global probe: flattening 150,528 dims into one# ridge against 32x32x3 is both intractable and the wrong question for it. Its proper# form is the per-patch probe in the next cell, where each token predicts its own block.REPS = {'farl_patch_mean': PMEAN}if POOLED is not None: REPS['pooled_state'] = POOLEDpath0_global = {}rng_ctrl = np.random.default_rng(4242)for name, Xall in REPS.items():    Xtr, Xte = Xall[TR_ROWS], Xall[TE_ROWS]    ctrl = rng_ctrl.normal(size=Xall.shape).astype(np.float32)    for label, A, B_ in ((name, Xtr, Xte), (f'random_ctrl::{name}', ctrl[TR_ROWS], ctrl[TE_ROWS])):        l2 = cv_l2(A, Y_tr, L2_GRID)        mu, sd = A.mean(0, keepdims=True), A.std(0, keepdims=True) + 1e-6        W, b = ridge_fit((A-mu)/sd, Y_tr, l2)        pred = ridge_predict((B_-mu)/sd, W, b)        r2 = r2_vs_mean(pred, Y_te, Y_tr_mean)        path0_global[label] = {'r2_vs_mean_image': r2, 'l2': float(l2), 'dims': int(A.shape[1])}        print(f'{label:34s} dims {A.shape[1]:6d}  l2 {l2:9.1f}  R2 vs mean-image {r2:+.4f}')print('\nR2 = 0 means "no better than predicting the training-block average frame".')

In [ ]:
# --- per-patch probe on dense tokens: token p -> its own 16x16x3 block --------def patch_targets(rows, p):    r0, c0 = (p // GRID) * PATCH, (p % GRID) * PATCH    out = np.empty((len(rows), PATCH * PATCH * 3), np.float32)    for i, r in enumerate(rows):        out[i] = np.asarray(PIX[r][r0:r0+PATCH, c0:c0+PATCH], np.float32).ravel() / 255.0    return outL2_PATCH = 1e3            # fixed; the CV sweep over 196 patches is not worth the minutessse_dense = sse_mean = 0.0sse_shuf = 0.0recon_te = np.zeros((len(TE_ROWS), IMG, IMG, 3), np.float32)rng_sh = np.random.default_rng(7)shuf_rows = rng_sh.permutation(TR_ROWS)for p in tqdm(range(GRID * GRID), desc='per-patch ridge'):    Xtr = np.asarray(DENSE[TR_ROWS, p, :], np.float32)    Xte = np.asarray(DENSE[TE_ROWS, p, :], np.float32)    Ytr, Yte = patch_targets(TR_ROWS, p), patch_targets(TE_ROWS, p)    mu, sd = Xtr.mean(0, keepdims=True), Xtr.std(0, keepdims=True) + 1e-6    W, b = ridge_fit((Xtr-mu)/sd, Ytr, L2_PATCH)    pred = ridge_predict((Xte-mu)/sd, W, b)    sse_dense += ((pred - Yte) ** 2).sum()    sse_mean  += ((Ytr.mean(0)[None] - Yte) ** 2).sum()    # shuffled-target control: same fit, permuted rows -> must land at the mean-image score    Ws, bs = ridge_fit((Xtr-mu)/sd, patch_targets(shuf_rows, p), L2_PATCH)    sse_shuf += ((ridge_predict((Xte-mu)/sd, Ws, bs) - Yte) ** 2).sum()    r0, c0 = (p // GRID) * PATCH, (p % GRID) * PATCH    recon_te[:, r0:r0+PATCH, c0:c0+PATCH, :] = pred.reshape(-1, PATCH, PATCH, 3)r2_patch = 1 - sse_dense / sse_meanr2_shuf  = 1 - sse_shuf / sse_meanpath0_patch = {'r2_vs_mean_image': float(r2_patch),               'shuffled_target_ctrl_r2': float(r2_shuf), 'l2': L2_PATCH}print(f'farl_dense per-patch ridge   R2 vs mean-image {r2_patch:+.4f}')print(f'  shuffled-target control    R2 vs mean-image {r2_shuf:+.4f}  (must be ~<= 0)')recon_te = np.clip(recon_te, 0, 1)

In [ ]:
k = np.linspace(0, len(TE_ROWS) - 1, 6).astype(int)fig, ax = plt.subplots(2, 6, figsize=(14, 5))for j, i in enumerate(k):    ax[0, j].imshow(np.asarray(PIX[TE_ROWS[i]])); ax[0, j].set_title(f'real {TEST_IDX[i]}', fontsize=9)    ax[1, j].imshow(recon_te[i]);                 ax[1, j].set_title('dense per-patch ridge', fontsize=9)    ax[0, j].axis('off'); ax[1, j].axis('off')plt.suptitle('Path 0 oracle reconstruction, held-out frames (closed-form, no training)')plt.tight_layout(); plt.show()

## 6. Path 1 — small trained decodersOne decoder per representation, all with the same budget and the same objective, sothe comparison is about the *latent* rather than about decoder tuning.- pooled / patch-mean → `Linear → 7×7×128 → 5× ConvTranspose → 224` (the architecture  already used in `JEPA_PooledState_Decoder_SanityCheck.ipynb`, so its numbers are  comparable — except that this one trains and tests on a contiguous split).- dense → `1×1 conv over the 14×14×768 grid → 4× ConvTranspose → 224`.L1 loss, and the **mean-image baseline is printed every epoch**. A decoder that doesnot clear it by a wide margin has learned to draw the average face.

In [ ]:
class VecDecoder(nn.Module):    def __init__(self, in_dim, base=128):        super().__init__()        self.base = base        self.fc = nn.Linear(in_dim, base * 7 * 7)        self.net = nn.Sequential(            nn.ConvTranspose2d(base, base, 4, 2, 1),        nn.BatchNorm2d(base),      nn.ReLU(True),            nn.ConvTranspose2d(base, base//2, 4, 2, 1),     nn.BatchNorm2d(base//2),   nn.ReLU(True),            nn.ConvTranspose2d(base//2, base//4, 4, 2, 1),  nn.BatchNorm2d(base//4),   nn.ReLU(True),            nn.ConvTranspose2d(base//4, base//8, 4, 2, 1),  nn.BatchNorm2d(base//8),   nn.ReLU(True),            nn.ConvTranspose2d(base//8, 3, 4, 2, 1),        nn.Sigmoid())    def forward(self, z):        return self.net(self.fc(z).view(-1, self.base, 7, 7))class DenseDecoder(nn.Module):    '''14x14x768 tokens -> 224x224x3. Keeps the spatial grid: this is the only    decoder here that can use WHERE information rather than only WHAT.'''    def __init__(self, tok=768, base=256):        super().__init__()        self.proj = nn.Conv2d(tok, base, 1)        self.net = nn.Sequential(            nn.ConvTranspose2d(base, base//2, 4, 2, 1),     nn.BatchNorm2d(base//2),   nn.ReLU(True),            nn.ConvTranspose2d(base//2, base//4, 4, 2, 1),  nn.BatchNorm2d(base//4),   nn.ReLU(True),            nn.ConvTranspose2d(base//4, base//8, 4, 2, 1),  nn.BatchNorm2d(base//8),   nn.ReLU(True),            nn.ConvTranspose2d(base//8, 3, 4, 2, 1),        nn.Sigmoid())    def forward(self, t):                     # t: (B, 196, 768)        x = t.transpose(1, 2).reshape(t.size(0), -1, GRID, GRID)        return self.net(self.proj(x))

In [ ]:
N_EPOCHS_DEC, BS_DEC, LR_DEC = 40, 32, 1e-3def get_pixels(rows):    a = np.asarray(PIX[rows], np.float32) / 255.0    return torch.from_numpy(a).permute(0, 3, 1, 2)MEAN_IMG = (np.asarray(PIX[TR_ROWS[::4]], np.float32) / 255.0).mean(0)      # train-block averagemean_img_t = torch.from_numpy(MEAN_IMG).permute(2, 0, 1).to(device)def train_decoder(name, latent_fn, dec, epochs=N_EPOCHS_DEC, tag=''):    dec = dec.to(device)    opt = torch.optim.Adam(dec.parameters(), lr=LR_DEC)    rng = np.random.default_rng(SEED)    for ep in range(epochs):        dec.train(); perm = rng.permutation(len(TR_ROWS)); tot = n = 0        for b0 in range(0, len(perm) - BS_DEC + 1, BS_DEC):            rows = TR_ROWS[perm[b0:b0 + BS_DEC]]            z = latent_fn(rows).to(device)            y = get_pixels(rows).to(device)            loss = F.l1_loss(dec(z), y)            opt.zero_grad(); loss.backward(); opt.step()            tot += loss.item() * len(rows); n += len(rows)        if (ep + 1) % 10 == 0 or ep == epochs - 1:            te = eval_decoder(dec, latent_fn)            print(f'  [{name}{tag}] ep {ep+1:3d}  train L1 {tot/n:.4f}  '                  f'test L1 {te["l1"]:.4f}  (mean-image {te["l1_mean_img"]:.4f})  '                  f'PSNR {te["psnr"]:.2f} dB (mean-image {te["psnr_mean_img"]:.2f})')    return dec@torch.no_grad()def eval_decoder(dec, latent_fn, rows=None):    dec.eval()    rows = TE_ROWS if rows is None else rows    l1 = l1m = mse = msem = 0.0; n = 0    for b0 in range(0, len(rows), 64):        r = rows[b0:b0 + 64]        y = get_pixels(r).to(device)        p = dec(latent_fn(r).to(device))        l1  += F.l1_loss(p, y, reduction='sum').item()        l1m += F.l1_loss(mean_img_t.expand_as(y), y, reduction='sum').item()        mse += F.mse_loss(p, y, reduction='sum').item()        msem+= F.mse_loss(mean_img_t.expand_as(y), y, reduction='sum').item()        n += y.numel()    return {'l1': l1/n, 'l1_mean_img': l1m/n,            'psnr': 10*math.log10(1.0/(mse/n)), 'psnr_mean_img': 10*math.log10(1.0/(msem/n))}LATENTS = {'farl_patch_mean': (lambda rows: torch.from_numpy(PMEAN[rows]).float(),                               lambda: VecDecoder(768))}if POOLED is not None:    LATENTS['pooled_state'] = (lambda rows: torch.from_numpy(POOLED[rows]).float(),                               lambda: VecDecoder(EMBED_DIM))LATENTS['farl_dense'] = (lambda rows: torch.from_numpy(np.asarray(DENSE[rows], np.float32)),                         lambda: DenseDecoder())DECODERS, path1 = {}, {}for name, (lat, mk) in LATENTS.items():    print(f'=== {name} ===')    d = train_decoder(name, lat, mk())    DECODERS[name] = d    path1[name] = eval_decoder(d, lat)    gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
print(f'{"representation":22s} {"dims":>8s} {"test L1":>9s} {"mean-img":>9s} {"gain":>7s} '      f'{"PSNR":>7s} {"mean-img":>9s}')for name, m in path1.items():    dims = {'pooled_state': EMBED_DIM, 'farl_patch_mean': 768, 'farl_dense': 150528}[name]    print(f'{name:22s} {dims:8d} {m["l1"]:9.4f} {m["l1_mean_img"]:9.4f} '          f'{100*(1-m["l1"]/m["l1_mean_img"]):6.1f}% {m["psnr"]:7.2f} {m["psnr_mean_img"]:9.2f}')print('\n"gain" is the L1 improvement over drawing the training-block average face.')print('Under ~15% and the reconstruction is essentially that average, whatever it looks like.')

## 7. The decisive check — N-way mouth-region matchingPixel metrics reward blur. This does not: it asks whether the reconstruction of frame*i* is closer to the **real** mouth crop of frame *i* than to 74 other real mouth cropsfrom the same held-out block. An average face scores at chance no matter how good itsL1 is.Null: Binomial(300, 1/75) — mean 4.000, SD 1.987. The p-value is the exact binomialsurvival function, **not** a normal tail on `(x−4)/1.987`; that approximation was usedbefore and is anti-conservative by up to 30× at these counts.Two controls run alongside: **latent-swap** (decode frame *j*, score against *i* — ifthis barely drops, the decoder ignores its input) and **mean-image** (which must landat chance, and is the sanity check on the harness itself).

In [ ]:
from scipy.stats import binom@torch.no_grad()def nway_match(recon_imgs, real_rows, n_queries=N_QUERIES, n_way=N_WAY, seed=SEED,               exclude_radius=0):    '''recon_imgs: (N, 224, 224, 3) float [0,1] aligned with real_rows.    Two details that matter, both found by smoke-testing this harness against a    mean-image control that should sit at chance and did not:    1. The true candidate is placed at a RANDOM position, not position 0. With the       true frame always first, np.argmin's first-wins tie-breaking hands every tie       to the correct answer - which pushed a mean-image control to 3x chance on       synthetic data. Exact pixel ties are unlikely on real frames, but near-ties       are not, and the fix is free.    2. exclude_radius drops frames within +/- r of the query from the distractor       pool. At 25 fps, frame i+-1 is nearly identical to frame i, so counting a       match to an adjacent frame as a MISS makes the test harsher than the question       being asked. exclude_radius=0 (the pre-registered primary) keeps it harsh;       the r=5 variant is reported alongside as a secondary.    '''    rng = np.random.default_rng(seed)    real = np.asarray(PIX[real_rows], np.float32) / 255.0    real_m = mouth_crop(real)    rec_m  = mouth_crop(recon_imgs)    N = len(real_rows)    n_queries = min(n_queries, N)    q = rng.choice(N, size=n_queries, replace=False)    hits, skipped = 0, 0    for i in q:        pool = np.arange(N)        pool = pool[np.abs(pool - i) > exclude_radius]        if len(pool) < n_way - 1:            skipped += 1; continue        distract = rng.choice(pool, size=n_way - 1, replace=False)        cand = np.concatenate([[i], distract])        order = rng.permutation(n_way)          # true candidate no longer always first        cand = cand[order]        true_pos = int(np.where(order == 0)[0][0])        d = ((real_m[cand] - rec_m[i][None]) ** 2).sum(1)        hits += int(np.argmin(d) == true_pos)    n_eff = n_queries - skipped    p = float(binom.sf(hits - 1, n_eff, 1.0 / n_way))    return {'hits': int(hits), 'n_queries': int(n_eff), 'n_way': int(n_way),            'exclude_radius': int(exclude_radius),            'expected_chance': n_eff / n_way, 'p_exact_binomial': p}@torch.no_grad()def reconstruct(dec, latent_fn, rows, swap=False, seed=SEED):    if swap:        rows_lat = np.random.default_rng(seed).permutation(rows)    else:        rows_lat = rows    out = np.empty((len(rows), IMG, IMG, 3), np.float32)    dec.eval()    for b0 in range(0, len(rows), 64):        z = latent_fn(rows_lat[b0:b0+64]).to(device)        out[b0:b0+64] = dec(z).permute(0, 2, 3, 1).cpu().numpy()    return np.clip(out, 0, 1)

In [ ]:
RECONS = {}decisive, decisive_r5 = {}, {}for name, (lat, _) in LATENTS.items():    rec = reconstruct(DECODERS[name], lat, TE_ROWS)    RECONS[name] = rec    decisive[name]    = nway_match(rec, TE_ROWS, exclude_radius=0)    decisive_r5[name] = nway_match(rec, TE_ROWS, exclude_radius=5)    swp = reconstruct(DECODERS[name], lat, TE_ROWS, swap=True)    decisive[f'{name}::latent_swap_ctrl'] = nway_match(swp, TE_ROWS)decisive['path0_dense_ridge']    = nway_match(recon_te, TE_ROWS, exclude_radius=0)decisive_r5['path0_dense_ridge'] = nway_match(recon_te, TE_ROWS, exclude_radius=5)MEAN_STACK = np.repeat(MEAN_IMG[None], len(TE_ROWS), 0)decisive['mean_image_ctrl'] = nway_match(MEAN_STACK, TE_ROWS)# harness self-test: a perfect reconstruction must be near-ceiling, the mean image at# chance. If either fails, the numbers below are about the harness, not the decoders.perfect_check = nway_match(np.asarray(PIX[TE_ROWS], np.float32) / 255.0, TE_ROWS)print(f'harness self-test: perfect recon {perfect_check["hits"]}/{perfect_check["n_queries"]} '      f'(must be near ceiling) | mean-image {decisive["mean_image_ctrl"]["hits"]}'      f'/{decisive["mean_image_ctrl"]["n_queries"]} (must be near '      f'{decisive["mean_image_ctrl"]["expected_chance"]:.1f})')assert perfect_check['hits'] > 0.8 * perfect_check['n_queries'], 'harness broken: perfect recon fails'print()print(f'{"condition":42s} {"hits":>5s} {"chance":>7s} {"p (exact)":>11s}  verdict')for k, v in decisive.items():    verdict = ('PASS  p<0.001' if v['hits'] >= 12 else               'weak  p<0.05'  if v['hits'] >= 8  else 'at chance')    print(f'{k:42s} {v["hits"]:5d} {v["expected_chance"]:7.2f} {v["p_exact_binomial"]:11.2e}  {verdict}')print(f'\nsecondary, distractors within +/-5 frames excluded (an easier, arguably fairer test):')for k, v in decisive_r5.items():    print(f'{k:42s} {v["hits"]:5d} {v["expected_chance"]:7.2f} {v["p_exact_binomial"]:11.2e}')

In [ ]:
# Confidence interval on the best condition - a bare count invites over-reading.from statsmodels.stats.proportion import proportion_confintbest_name = max((k for k in decisive if 'ctrl' not in k), key=lambda k: decisive[k]['hits'])b = decisive[best_name]lo, hi = proportion_confint(b['hits'], b['n_queries'], alpha=0.05, method='beta')print(f'best condition: {best_name}  {b["hits"]}/{b["n_queries"]} = {b["hits"]/b["n_queries"]:.4f}')print(f'95% CI on the accuracy: [{lo:.4f}, {hi:.4f}]   (chance = {1/N_WAY:.4f} = {1/N_WAY*100:.2f}%)')print()print('Power, for reading a null: MDE at 80% power is 2.54x chance (~10 hits). Power is')print('25% at 1.5x chance and 55% at 2x. P(1 <= X <= 8) = 0.9616 under the null - so a')print('single-digit count is ordinary noise and does NOT establish "no effect".')

## 8. Reconstructed video of the held-out block

In [ ]:
VIDEO_REP = best_name if best_name in LATENTS else 'farl_patch_mean'rec = RECONS[VIDEO_REP]real = np.asarray(PIX[TE_ROWS], np.float32) / 255.0out_path = os.path.join(RESULTS_DIR, f'{IDENTITY}_heldout_oracle_{VIDEO_REP}.mp4')vw = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), 25.0, (IMG * 2, IMG))for i in range(len(rec)):    side = np.concatenate([real[i], rec[i]], axis=1)    vw.write(cv2.cvtColor((side * 255).astype(np.uint8), cv2.COLOR_RGB2BGR))vw.release()print(f'wrote {out_path}  ({len(rec)} frames = {len(rec)/25:.1f} s, real | reconstruction)')# mux the real audio over it so lip-sync can be judged by ear, not only by metricaud_wav = os.path.join(LOCAL_PATH, 'aud.wav')if os.path.isfile(aud_wav):    t0 = TEST_IDX.min() / 25.0    av_path = out_path.replace('.mp4', '_audio.mp4')    import subprocess    r = subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', out_path,                        '-ss', f'{t0:.3f}', '-i', aud_wav,                        '-c:v', 'copy', '-c:a', 'aac', '-shortest', av_path],                       capture_output=True, text=True)    print(f'wrote {av_path}  (test block starts at t={t0:.2f}s in aud.wav)'          if r.returncode == 0 else f'ffmpeg failed: {r.stderr[:400]}')k = np.linspace(0, len(rec) - 1, 8).astype(int)fig, ax = plt.subplots(2, 8, figsize=(16, 4.4))for j, i in enumerate(k):    ax[0, j].imshow(real[i]); ax[0, j].axis('off')    ax[1, j].imshow(rec[i]);  ax[1, j].axis('off')ax[0, 0].set_ylabel('real'); ax[1, 0].set_ylabel('recon')plt.suptitle(f'held-out oracle reconstruction - {VIDEO_REP}')plt.tight_layout(); plt.show()

## 9. Verdict against the pre-registration

In [ ]:
main = {k: v for k, v in decisive.items() if 'ctrl' not in k}best_hits = max(v['hits'] for v in main.values())winner = max(main, key=lambda k: main[k]['hits'])if best_hits >= 12:    verdict = 'PASS'    action = (f'{winner} clears p<0.001. The latent is invertible enough to carry per-frame '              f'mouth configuration. Next: decode PREDICTED latents (not oracle ones) and '              f'repeat this exact check - that is the real target, and the gap between the '              f'two numbers is the paper.')elif best_hits >= 8:    verdict = 'WEAK'    action = (f'{winner} clears p<0.05 but not p<0.001. Real but small. Before scaling the '              f'decoder, check whether dense tokens beat the pooled vector here - if they do, '              f'the mean-pool in FarlFrozenFaceEncoder is the bottleneck and the fix is one '              f'line (return x[:, 1:] instead of its mean) plus a wider projector.')else:    verdict = 'FAIL'    action = ('No representation clears p<0.05. Per the pre-registered stop rule: report this '              'and stop - do NOT escalate to a diffusion decoder. The previous diffusion arc '              'already produced 1/300 at chance and cost weeks. A negative invertibility result '              'on a pooled JEPA latent, with dense tokens as the measured contrast, is itself '              'a publishable finding and is what the field currently lacks.')RESULT = {    'study_id': STUDY_ID, 'run_tag': RUN_TAG,    'evaluated_at_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),    'prereg_registered_at_utc': PREREG['registered_at_utc'],    'mouth_box': {'top_left': [int(MOUTH[0]), int(MOUTH[1])], 'size': [H, W]},    'path0_global': path0_global, 'path0_dense_per_patch': path0_patch,    'path1_trained_decoders': path1,    'decisive_check': decisive,    'decisive_check_exclude_r5': decisive_r5,    'harness_self_test_perfect_recon': perfect_check,    'best_condition': winner, 'best_hits': int(best_hits),    'verdict': verdict,}path = os.path.join(RESULTS_DIR, f'{STUDY_ID}_{RUN_TAG}.json')with open(path, 'w') as f: json.dump(RESULT, f, indent=2)print('=' * 78)print(f'PRIMARY ENDPOINT  N-way mouth matching, held-out, oracle latent')print(f'best              {winner}: {best_hits}/{N_QUERIES} (chance {N_QUERIES/N_WAY:.1f})')print(f'VERDICT           {verdict}')print('=' * 78)print(action)print(f'\nwritten: {path}')

## 10. What each result means for the project**The comparison that matters is between rows, not the absolute numbers.**- `farl_dense` ≫ `farl_patch_mean` → the mean-pool is the bottleneck. This is the  cheapest possible fix and the clearest story: one line in the encoder  (`return x[:, 1:]`), a projector that takes 196×768, and the predictor operating on  spatial tokens. It also puts the work back in the RAE / VideoRAE regime where the  published reconstruction numbers actually apply.- `farl_patch_mean` ≈ `farl_dense`, both good → pooling is not costing much and the  256-d `pooled_state` is the place to look next.- `pooled_state` ≪ `farl_patch_mean` → the *projector* is discarding what a decoder  needs, i.e. the JEPA objective is not preserving decodable content. That is a real  finding about JEPA-for-synthesis and is the most interesting negative result  available here.- all three at chance → the stop rule fires. Write it up as a negative result.**Keep the oracle/predicted distinction visible in everything you write.** Thisnotebook measures the *encoder's* invertibility. A watchable video from oraclelatents is not talking-head synthesis — it is a compression result. Synthesis needsthe predicted latent to clear the same bar, and the honest headline is always thepair of numbers.**Before writing "to our knowledge, the first":** the narrow claim (a JEPA applied toaudio-driven talking-head synthesis) survived checking against the 11 competitorsystems, none of which is a JEPA. Do one Google Scholar pass immediately beforesubmission anyway. Also still open: FaRL's MIM target type (pixels vs discretetokens) is unverified, and V-JEPA's appendix (arXiv 2404.08471) should be checked fora diffusion decoder before claiming novelty on latent→RGB for JEPAs.**Do not release person-specific checkpoints.** Code and the label pipeline, yes; theidentity-specific weights are the directly misusable artifact. The institution makesthe IRB determination, not this notebook.